# FingerBall Stage 2J-1 — Multi-model 2D / 3D Hand Comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_Stage2J1_MultiModel_2D3D_Comparison.ipynb)

### Goal
**Stage 2Hの目視確認済みXYを変更せず**、MediaPipe Tasks Hand LandmarkerとWiLoR-miniを同じ元動画フレームに推論させ、2D画素座標、推定3D、手の左右、モデル間の誤差を統一表で比較します。

- 元動画 1206×648 / frame 185–198、特に188–193（190はリリース目視参照、未自動検出）。
- 2D座標はすべて**元動画のピクセル座標**へ変換。モデルが異なる3D座標は**生値を別の座標系として記録**し、直接平均しません。
- 入力モデルの選択について**手動ROIを使用する条件（oracle-assisted）**と、モデル独立検出の条件を区別します。ROI assistedの2D誤差は**独立した検出性能の評価ではありません**。
- 遮蔽点は評価から除外。手動点、目視確認済み引継ぎ候補、AI点の出自を保持。
- 失敗した検出は欠損として出力し、成功したように補間しません。
- WiLoR-miniは内部で**MANO_RIGHT.pklをHugging Faceから自動取得**する実装です。MANOのモデルライセンスを必ず確認し、許諾を得るまで`ENABLE_WILOR=False`で実行してください。<https://mano.is.tue.mpg.de/>

参考：<https://github.com/warmshao/WiLoR-mini>、<https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker/python>

**これは比較研究用Notebookで、SOMAフィッティングは次のStage 2J-2です。**

## 0. Colabセットアップ：MediaPipe単独は常に実行可能

WiLoR-miniは旧Python 3.10想定で、最新Colabでは依存ライブラリとMANOライセンスの問題があり得ます。比較表はWiLoRが動かない場合も生成します。

In [ ]:
%pip -q install "mediapipe>=0.10.20" "opencv-python-headless>=4.8" "pandas>=2" "numpy>=1.24" "matplotlib>=3.7" "plotly>=5.18"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib,json,time,traceback,urllib.request,subprocess,sys
import numpy as np,pandas as pd,cv2,matplotlib.pyplot as plt
import mediapipe as mp
from IPython.display import display
print('Python',sys.version.split()[0],'MediaPipe',mp.__version__)

## 1. 設定・元データの読み込み

In [ ]:
PROJECT=Path('/content/drive/MyDrive/母指回内トラッキング')
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
VIDEO_PATH=PROJECT/(VIDEO_STEM+'.mp4')  # 加工済みzoom動画は使わない
STAGE2H_CSV=PROJECT/'FingerBall_v2_stage2h_results'/VIDEO_STEM/'hand_21_landmarks_2d_corrected.csv'
OUTPUT_DIR=PROJECT/'FingerBall_v2_stage2j1_results'/VIDEO_STEM
FRAME_START,FRAME_END=185,198
RELEASE_REFERENCE=190
THROWING_HAND='Left'
USE_MANUAL_ROI_FOR_MEDIAPIPE=True
ENABLE_WILOR=False   # MANOのライセンス・配布物の利用許諾を確認した場合のみ True
I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS=False
WILOR_RUN_DETECTOR_BASELINE=True
WILOR_RUN_MANUAL_BBOX_ASSIST=True
ROI_MARGIN=1.8
FRAME_DISPLAY=(188,189,190,191,192,193)
if THROWING_HAND not in ('Left','Right'):raise ValueError('Throwing hand must be Left/Right')
if ENABLE_WILOR and not I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS:
    raise PermissionError('WiLoR-miniはMANOモデルファイルを自動取得します。MANOの利用許諾を確認してからフラグを設定してください')
for file in (VIDEO_PATH,STAGE2H_CSV):
    if not file.is_file():raise FileNotFoundError(file)
if 'zoom' in VIDEO_PATH.stem.lower():raise ValueError('元動画を指定してください')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
source_hash=hashlib.sha256(STAGE2H_CSV.read_bytes()).hexdigest()
ref=pd.read_csv(STAGE2H_CSV)
req={'frame','landmark_id','x_px','y_px','annotation_status'}
if diff:=req-set(ref.columns):raise ValueError(f'Missing Stage2H columns: {diff}')
if ref.duplicated(['frame','landmark_id']).any():raise ValueError('Duplicate rows')
frames=list(range(FRAME_START,FRAME_END+1))
ref=ref[ref.frame.isin(frames)].copy()
assert len(ref)==21*len(frames)
assert all(ref.groupby('frame').landmark_id.nunique().eq(21))
cap=cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():raise RuntimeError('Video failed to open')
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
assert FRAME_END<int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
assert W>0 and H>0
def frame_rgb(f):
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
    ok,bgr=cap.read()
    if not ok:raise RuntimeError(f'Could not read frame {f}')
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
images={f:frame_rgb(f) for f in frames}
cap.release()
reference={};status={}
for f in frames:
    a=ref[ref.frame==f].set_index('landmark_id').reindex(range(21))
    reference[f]=a[['x_px','y_px']].to_numpy(float)
    status[f]=a.annotation_status.astype(str).to_list()
print('Video',W,H,'FPS',FPS,'Frozen SHA256',source_hash)
print('Annotations',ref.annotation_status.value_counts().to_dict())

## 2. ROIと共通フォーマット

手動点に基づくROIは研究上**手動補助条件**です。独立した自動検出性能の評価と区別します。

In [ ]:
EDGE_SETS={'Thumb':[0,1,2,3,4],'Index':[0,5,6,7,8],
           'Middle':[0,9,10,11,12],'Ring':[0,13,14,15,16],
           'Little':[0,17,18,19,20]}
COLORS={'Thumb':'#d97706','Index':'#0891b2','Middle':'#be185d',
        'Ring':'#94a3b8','Little':'#64748b'}
NAME=['WRIST','THUMB_CMC','THUMB_MCP','THUMB_IP','THUMB_TIP',
      'INDEX_MCP','INDEX_PIP','INDEX_DIP','INDEX_TIP',
      'MIDDLE_MCP','MIDDLE_PIP','MIDDLE_DIP','MIDDLE_TIP',
      'RING_MCP','RING_PIP','RING_DIP','RING_TIP',
      'PINKY_MCP','PINKY_PIP','PINKY_DIP','PINKY_TIP']
def manual_box(f,margin=ROI_MARGIN):
    p=reference[f]
    p=p[np.isfinite(p).all(axis=1)]
    if len(p)<3:raise ValueError(f'Frame {f}: too few reference points for assisted ROI')
    lo=np.min(p,axis=0);hi=np.max(p,axis=0)
    center=(lo+hi)/2
    side=max(float(max(hi-lo))*margin,100.)
    x1=max(0,center[0]-side/2);y1=max(0,center[1]-side/2)
    x2=min(W,center[0]+side/2);y2=min(H,center[1]+side/2)
    if x2<=x1 or y2<=y1:raise ValueError('Invalid box')
    return np.array([x1,y1,x2,y2],float)
def image_crop(image,box,size=512):
    x1,y1,x2,y2=box
    side=max(x2-x1,y2-y1);cx=(x1+x2)/2;cy=(y1+y2)/2
    scale=size/side
    M=np.float32([[scale,0,size/2-cx*scale],[0,scale,size/2-cy*scale]])
    patch=cv2.warpAffine(image,M,(size,size),flags=cv2.INTER_LINEAR,
               borderMode=cv2.BORDER_CONSTANT,borderValue=(105,105,105))
    Minv=cv2.invertAffineTransform(M)
    return patch,Minv
def map_to_original(p,Minv):
    return np.concatenate([p,np.ones((len(p),1))],axis=1)@Minv.T
empty21=lambda:np.full((21,3),np.nan,float)
predictions={}
def store(f,method,xy=None,world=None,relative=None,handedness='',confidence=np.nan,
          detected=False,reason='',assisted=False):
    if xy is None:xy=np.full((21,2),np.nan)
    if world is None:world=empty21()
    if relative is None:relative=np.full((21,),np.nan)
    xy=np.asarray(xy,dtype=float).reshape(21,2)
    world=np.asarray(world,dtype=float).reshape(21,3)
    relative=np.asarray(relative,dtype=float).reshape(21)
    predictions[(int(f),method)]=dict(xy=xy,world=world,relative=relative,
         handedness=str(handedness),confidence=float(confidence),
         detected=bool(detected),reason=str(reason),assisted=bool(assisted))
def choose_hand(candidates,refxy,requested='Left'):
    # candidate dict: xy + hand label, score
    if not candidates:return None
    ranked=[]
    for c in candidates:
        good=np.isfinite(refxy).all(axis=1)&np.isfinite(c['xy']).all(axis=1)
        dist=float(np.median(np.linalg.norm(refxy[good]-c['xy'][good],axis=1))) if np.any(good) else 1e7
        handedness_penalty=0 if c.get('hand','')==requested else 200
        ranked.append((dist+handedness_penalty,c))
    return min(ranked,key=lambda v:v[0])[1]

## 3. MediaPipe Tasks：ROI補助と独立全画面検出の両方

画像座標に変換した21点、相対Z、model-estimated world 3Dを保存します。World 3Dは物理的なground truthではありません。

In [ ]:
task_path=PROJECT/'models'/'hand_landmarker.task'
task_path.parent.mkdir(parents=True,exist_ok=True)
if not task_path.is_file():
    url='https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
    urllib.request.urlretrieve(url,task_path)
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
mp_opts=mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(task_path)),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=2,min_hand_detection_confidence=.15,min_hand_presence_confidence=.15)
def infer_mp(img,offset_transform=None):
    mp_image=mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(img))
    result=landmarker.detect(mp_image)
    candidates=[]
    for k,landmarks in enumerate(result.hand_landmarks):
        cxy=np.array([[p.x*img.shape[1],p.y*img.shape[0]] for p in landmarks])
        xy=map_to_original(cxy,offset_transform) if offset_transform is not None else cxy
        z=np.array([p.z for p in landmarks])
        world=np.array([[p.x,p.y,p.z] for p in result.hand_world_landmarks[k]]) if k<len(result.hand_world_landmarks) else empty21()
        h=result.handedness[k][0] if k<len(result.handedness) and result.handedness[k] else None
        candidates.append({'xy':xy,'relative':z,'world':world,'hand':h.category_name if h else '',
                           'score':float(h.score) if h else np.nan})
    return candidates
with mp_vision.HandLandmarker.create_from_options(mp_opts) as landmarker:
    for f in frames:
        # Independent full-frame baseline: no manual ROI.
        try:
            candidates=infer_mp(images[f])
            chosen=choose_hand(candidates,reference[f],THROWING_HAND)
            if chosen is None:store(f,'mediapipe_full',reason='no_hand')
            else:store(f,'mediapipe_full',xy=chosen['xy'],world=chosen['world'],
                   relative=chosen['relative'],handedness=chosen['hand'],
                   confidence=chosen['score'],detected=True)
        except Exception as e:
            store(f,'mediapipe_full',reason=f'{type(e).__name__}:{e}')
        if USE_MANUAL_ROI_FOR_MEDIAPIPE:
            try:
                patch,inv=image_crop(images[f],manual_box(f))
                chosen=choose_hand(infer_mp(patch,inv),reference[f],THROWING_HAND)
                if chosen is None:store(f,'mediapipe_manual_roi',reason='no_hand',assisted=True)
                else:store(f,'mediapipe_manual_roi',xy=chosen['xy'],world=chosen['world'],
                      relative=chosen['relative'],handedness=chosen['hand'],
                      confidence=chosen['score'],detected=True,assisted=True)
            except Exception as e:
                store(f,'mediapipe_manual_roi',reason=f'{type(e).__name__}:{e}',assisted=True)
print('MediaPipe results:',{name:sum(predictions[(f,name)]['detected'] for f in frames)
    for name in ('mediapipe_full','mediapipe_manual_roi') if (frames[0],name) in predictions})

## 4. WiLoR-mini（Python 3.10独立環境、任意）

Colab本体はPython 3.13のままで、`uv`を用いてWiLoR-mini専用のPython 3.10仮想環境を作成します。WiLoR-miniの旧依存ライブラリがMediaPipe環境を壊さない構成です。初回はPyTorch等の依存ライブラリをインストールする必要があります。

**重要**：WiLoR-miniはMANOモデルを自動ダウンロードします。利用許諾を確認済みの場合のみ `ENABLE_WILOR=True` と `I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS=True` を設定してください。


In [ ]:
if ENABLE_WILOR:
    if not I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS:
        raise PermissionError('MANOモデルの利用許諾を確認する必要があります')

    # WiLoR-mini's released requirements specify torch<=2.5 and
    # ultralytics==8.1.34, which are unsuitable for Colab Python 3.13.
    # Keep the current notebook kernel untouched; use an isolated Python 3.10 worker.
    import shutil
    WILOR_ENV=Path('/content/wilor_py310')
    WILOR_PY=WILOR_ENV/'bin'/'python'
    WILOR_CACHE=OUTPUT_DIR/'wilor_worker_io'
    WILOR_CACHE.mkdir(parents=True,exist_ok=True)

    def run_logged(args,step):
        print('\n[WiLoR setup]',step,flush=True)
        result=subprocess.run([str(v) for v in args],text=True,
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        print(result.stdout[-6000:],flush=True)
        if result.returncode:
            raise RuntimeError(
                f'{step} failed (return code {result.returncode}). '
                'インストールログ末尾を確認してください。\n'+result.stdout[-7000:])

    if not WILOR_PY.is_file():
        # uv can download Python 3.10 independently of Colab's Python 3.13.
        if shutil.which('uv') is None:
            run_logged([sys.executable,'-m','pip','install','-q','uv'],'Install uv')
        uv=shutil.which('uv') or str(Path(sys.executable).parent/'uv')
        if not Path(uv).is_file() and not shutil.which('uv'):
            raise RuntimeError('uv binary was not installed; restart and retry')
        run_logged([uv,'python','install','3.10'],'Download isolated Python 3.10')
        run_logged([uv,'venv','--python','3.10',str(WILOR_ENV)],'Create Python 3.10 environment')
    else:
        uv=shutil.which('uv') or str(Path(sys.executable).parent/'uv')

    # Use a versioned environment marker: installations need only occur once.
    marker=WILOR_ENV/'fingerball_setup_v2.ok'
    if not marker.is_file():
        run_logged([uv,'pip','install','--python',str(WILOR_PY),
             'torch==2.5.1','torchvision==0.20.1','numpy==1.26.4',
             'opencv-python-headless==4.10.0.84','scipy<1.15',
             'setuptools<81','wheel','pip'],
            'Install compatible PyTorch and numerical runtime')
        run_logged([uv,'pip','install','--python',str(WILOR_PY),
             'smplx==0.1.28','timm','einops','roma',
             'huggingface_hub','scikit-image<0.26',
             'ultralytics==8.1.34','tqdm',
             'chumpy @ git+https://github.com/mattloper/chumpy'],
            'Install WiLoR-mini dependencies')
        run_logged([uv,'pip','install','--python',str(WILOR_PY),
             '--no-deps','git+https://github.com/warmshao/WiLoR-mini'],
            'Install WiLoR-mini without incompatible upstream dependency resolution')
        marker.write_text('Python 3.10 isolated WiLoR-mini environment',encoding='utf-8')

    # Pass frames and manual bounding boxes as an NPZ file; avoid passing
    # user-supplied data or large arrays through command-line arguments.
    import textwrap
    inp=WILOR_CACHE/'wilor_input.npz'
    out=WILOR_CACHE/'wilor_output.npz'
    frames_rgb=np.stack([images[f] for f in frames])
    bboxes=np.stack([manual_box(f) for f in frames])
    np.savez_compressed(inp,frames=np.asarray(frames),images=frames_rgb,
                        bboxes=bboxes,
                        hand_flag=np.array([1 if THROWING_HAND=='Right' else 0]))
    if out.exists():out.unlink()

    worker=WILOR_CACHE/'run_wilor_worker.py'
    worker.write_text(textwrap.dedent(r"""
    import sys,os,inspect
    # Compatibility shims for legacy chumpy on modern NumPy/Python.
    import numpy as np
    if not hasattr(inspect,'getargspec'):
        inspect.getargspec=inspect.getfullargspec
    for k,v in {'bool':bool,'int':int,'float':float,'complex':complex,
                'object':object,'str':str,'unicode':str}.items():
        if k not in np.__dict__:setattr(np,k,v)
    import torch
    from wilor_mini.pipelines.wilor_hand_pose3d_estimation_pipeline import (
        WiLorHandPose3dEstimationPipeline)
    a=np.load(sys.argv[1])
    frames=a['frames'].tolist()
    images=a['images']
    bboxes=a['bboxes']
    hand_flag=int(a['hand_flag'][0])
    dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('WiLoR isolated Python',sys.version.split()[0],
          'torch',torch.__version__,'device',dev,flush=True)
    model=WiLorHandPose3dEstimationPipeline(
        device=dev,dtype=torch.float16 if dev.type=='cuda' else torch.float32,
        verbose=False)
    records=[]
    def record(frame,mode,out,missing=''):
        d={'frame':frame,'mode':mode,'detected':False,
           'xy':np.full((21,2),np.nan),'xyz':np.full((21,3),np.nan),
           'hand':'','reason':missing}
        if out is not None and 'wilor_preds' in out:
            pred=out['wilor_preds']
            xy=np.asarray(pred['pred_keypoints_2d'])[0,:,:2]
            xyz=np.asarray(pred['pred_keypoints_3d'])[0,:,:3]
            if xy.shape==(21,2) and xyz.shape==(21,3):
                d.update(detected=True,xy=xy,xyz=xyz,
                         hand='Right' if float(out['is_right'])>=.5 else 'Left',
                         reason='')
            else:d['reason']=f'bad joint shape {xy.shape} {xyz.shape}'
        records.append(d)
    for j,f in enumerate(frames):
        img=np.ascontiguousarray(images[j])
        for mode in ('wilor_full_detector','wilor_manual_bbox'):
            try:
                if mode=='wilor_full_detector':
                    candidates=model.predict(img)
                    candidates=[o for o in candidates if 'wilor_preds' in o]
                    # If more than one hand appears, prioritize the requested side.
                    matching=[o for o in candidates if (float(o['is_right'])>=.5)==bool(hand_flag)]
                    selected=(matching or candidates)
                    found=selected[0] if selected else None
                else:
                    founds=model.predict_with_bboxes(img,
                          bboxes[j:j+1].astype(np.float32),
                          np.array([hand_flag],dtype=np.int64))
                    found=founds[0] if founds else None
                record(f,mode,found,missing='no_hand')
            except Exception as e:
                record(f,mode,None,missing=f'{type(e).__name__}:{e}')
        print('WiLoR processed frame',f,flush=True)
    np.savez_compressed(sys.argv[2],
        frame=np.array([r['frame'] for r in records],dtype=int),
        mode=np.array([r['mode'] for r in records],dtype=str),
        detected=np.array([r['detected'] for r in records],dtype=bool),
        xy=np.stack([r['xy'] for r in records]),
        xyz=np.stack([r['xyz'] for r in records]),
        hand=np.array([r['hand'] for r in records],dtype=str),
        reason=np.array([r['reason'] for r in records],dtype=str))
    """),encoding='utf-8')
    # Keep the worker's CUDA_VISIBLE_DEVICES inherited from Colab.
    run_logged([str(WILOR_PY),str(worker),str(inp),str(out)],
               'Run WiLoR-mini inference (isolated Python 3.10)')
    result=np.load(out)
    for k in range(len(result['frame'])):
        f=int(result['frame'][k]);method=str(result['mode'][k])
        if method=='wilor_full_detector' and not WILOR_RUN_DETECTOR_BASELINE:continue
        if method=='wilor_manual_bbox' and not WILOR_RUN_MANUAL_BBOX_ASSIST:continue
        success=bool(result['detected'][k])
        store(f,method,xy=result['xy'][k],world=result['xyz'][k],
              handedness=str(result['hand'][k]),detected=success,
              assisted=(method=='wilor_manual_bbox'),
              reason=str(result['reason'][k]))
else:
    print('WiLoR-mini disabled. MediaPipe comparisons proceed normally.')
    for f in frames:
        for label,assisted in [('wilor_full_detector',False),('wilor_manual_bbox',True)]:
            store(f,label,reason='disabled_or_license_not_confirmed',assisted=assisted)

print('WiLoR detection:',{
    method:sum(predictions[(f,method)]['detected'] for f in frames
               if (f,method) in predictions)
    for method in ('wilor_full_detector','wilor_manual_bbox')})

## 5. 共通21点CSVと比較指標

長形式：**1 frame×21 landmarks×method**、手動補正ステータスとXY、各予測XY、相対Z/推定3D（モデルごとの元座標・単位）、エラー。未検出は全21点をNaNにして行を維持します。

In [ ]:
methods=['stage2h_reference','mediapipe_full','mediapipe_manual_roi',
         'wilor_full_detector','wilor_manual_bbox']
rows=[]
for f in frames:
    for method in methods:
        r=predictions.get((f,method))
        if method=='stage2h_reference':
            # No 3D ground truth; only user-reviewed 2D.
            r=dict(xy=reference[f],world=empty21(),relative=np.full(21,np.nan),
                   handedness=THROWING_HAND,confidence=np.nan,detected=True,
                   assisted=False,reason='manual_reviewed_xy_only')
        if r is None:
            r=dict(xy=np.full((21,2),np.nan),world=empty21(),
                   relative=np.full(21,np.nan),handedness='',confidence=np.nan,
                   detected=False,assisted=False,reason='not_run')
        for i in range(21):
            p=r['xy'][i]; gt=reference[f][i]; q=r['world'][i]
            valid=np.isfinite(gt).all() and status[f][i] not in ('occluded','missing')
            err=float(np.linalg.norm(p-gt)) if valid and np.isfinite(p).all() and method!='stage2h_reference' else np.nan
            rows.append({'frame':f,'time_s':f/FPS,'method':method,'landmark_id':i,
              'name':NAME[i],'status_stage2h':status[f][i],
              'stage2h_x_px':gt[0] if valid else np.nan,
              'stage2h_y_px':gt[1] if valid else np.nan,
              'x_px':p[0],'y_px':p[1],
              'z_relative':r['relative'][i],
              'model3d_x':q[0],'model3d_y':q[1],'model3d_z':q[2],
              'model3d_coord_system':('mediapipe_model_world' if method.startswith('mediapipe_') else
                    'wilor_mano_pred_keypoints_3d' if method.startswith('wilor_') else 'none'),
              'detected':bool(r['detected']),'assisted_by_stage2h_roi':bool(r['assisted']),
              'handedness_reported':r['handedness'],'score_reported':r['confidence'],
              'error_to_stage2h_px':err,'failure_reason':r['reason']})
comparison=pd.DataFrame(rows)
common_csv=OUTPUT_DIR/'stage2j1_common_21_landmarks_with_provenance.csv'
comparison.to_csv(common_csv,index=False)
summary=[]
for method,g in comparison[comparison.method!='stage2h_reference'].groupby('method'):
    values=g.error_to_stage2h_px.dropna()
    vals_manual=g.loc[g.status_stage2h=='manual_visible','error_to_stage2h_px'].dropna()
    summary.append({'method':method,
        'detected_frames':int(g.groupby('frame').detected.first().sum()),
        'total_frames':len(frames),
        'n_points_evaluable':len(values),
        'RMSE_px':float(np.sqrt(np.mean(values**2))) if len(values) else np.nan,
        'median_px':float(values.median()) if len(values) else np.nan,
        'manual_only_n':len(vals_manual),
        'manual_only_RMSE_px':float(np.sqrt(np.mean(vals_manual**2))) if len(vals_manual) else np.nan,
        'assisted_with_reference_ROI':bool(g.assisted_by_stage2h_roi.any())})
summary=pd.DataFrame(summary)
summary.to_csv(OUTPUT_DIR/'stage2j1_model_comparison_summary.csv',index=False)
display(summary)
print('CSV:',common_csv)

## 6. 188–193フレームの2D比較図

モデル別の21点を元動画上に重畳して比較します。**ステータスの欠損・遮蔽点は描画しません。** 失敗したモデルには"NO DETECTION"と表示します。

In [ ]:
def draw_skeleton(ax,xy,color,linewidth=1.7,dotsize=13):
    for finger,ids in EDGE_SETS.items():
        for a,b in zip(ids[:-1],ids[1:]):
            if np.isfinite(xy[[a,b]]).all():
                ax.plot(xy[[a,b],0],xy[[a,b],1],color=color,lw=linewidth,alpha=.95)
    ok=np.isfinite(xy).all(axis=1)
    if ok.any():ax.scatter(xy[ok,0],xy[ok,1],c=color,s=dotsize,zorder=3)
plot_methods=['stage2h_reference','mediapipe_full','mediapipe_manual_roi',
               'wilor_full_detector','wilor_manual_bbox']
for f in FRAME_DISPLAY:
    if f not in frames:continue
    fig,axes=plt.subplots(1,len(plot_methods),figsize=(20,5),sharex=True,sharey=True)
    center=np.nanmedian(reference[f],axis=0)
    for ax,method in zip(axes,plot_methods):
        ax.imshow(images[f])
        if method=='stage2h_reference':
            coords=reference[f].copy()
            for i,st in enumerate(status[f]):
                if st in ('occluded','missing'):coords[i]=np.nan
            detected=True
        else:
            pred=predictions.get((f,method),{})
            coords=pred.get('xy',np.full((21,2),np.nan))
            detected=pred.get('detected',False)
        draw_skeleton(ax,coords,'#00e6ff' if method!='stage2h_reference' else '#ffb545')
        if not detected:
            ax.text(.5,.5,'NO DETECTION',ha='center',va='center',
                     transform=ax.transAxes,color='red',fontsize=13,
                     bbox={'facecolor':'white','alpha':.8})
        ax.set_title(method,fontsize=10)
        ax.set_xlim(max(0,center[0]-95),min(W,center[0]+95))
        ax.set_ylim(min(H,center[1]+95),max(0,center[1]-95))
        ax.set_xticks([]);ax.set_yticks([])
    fig.suptitle(f'Frame {f} — Stage 2H versus independently inferred and assisted models')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/f'stage2j1_frame_{f}_2d_comparison.png',dpi=145)
    plt.show()


## 7. 3D骨格の内部形状比較

MediaPipe world 3DとWiLoR 3Dは原点・スケール・座標軸が異なるため、直接の3D RMSEを報告しません。参考図では手首原点、MCP手掌スケールで正規化し、**表示用に正規化した形状**として並べます。これは3D精度検証ではありません。

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
def normalize_model_3d(x):
    x=np.array(x,float)
    if x.shape!=(21,3) or not np.isfinite(x).all():return None
    p=x-x[0]
    palm=float(np.linalg.norm(p[9]))
    if not np.isfinite(palm) or palm<1e-8:return None
    return p/palm
display_3d_methods=['mediapipe_full','mediapipe_manual_roi',
                    'wilor_full_detector','wilor_manual_bbox']
for f in (190,191,192):
    if f not in frames:continue
    fig=plt.figure(figsize=(14,3.6))
    for i,method in enumerate(display_3d_methods):
        ax=fig.add_subplot(1,4,i+1,projection='3d')
        data=predictions.get((f,method))
        a=normalize_model_3d(data['world']) if data is not None and data['detected'] else None
        if a is not None:
            for name,ids in EDGE_SETS.items():
                ax.plot(a[ids,0],a[ids,1],a[ids,2],lw=2,label=name)
            ax.scatter(a[:,0],a[:,1],a[:,2],s=8,c='black')
        else:ax.text2D(.2,.5,'NO 3D DATA',transform=ax.transAxes)
        ax.set(title=method,box_aspect=(1,1,1))
        ax.set(xlim=(-3,3),ylim=(-3,3),zlim=(-3,3))
    fig.suptitle(f'Frame {f}: normalized model-specific 3D (NOT 3D ground truth)',fontsize=11)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/f'stage2j1_frame_{f}_3d_comparison.png',dpi=145)
    plt.show()


## 8. 検出率・フレーム別2D誤差の可視化と再現性記録

「モデルが検出に成功した割合」と「検出した点がStage2Hの目視確認済み座標に近いか」を分けて集計します。同じ手動ROIを利用した条件は**oracle-assisted**と明記し、独立検出性能とはみなしません。

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
for method in ['mediapipe_full','mediapipe_manual_roi','wilor_full_detector','wilor_manual_bbox']:
    part=comparison[(comparison.method==method)&
         (comparison.status_stage2h=='manual_visible')]
    errs=part.groupby('frame').error_to_stage2h_px.agg(
        lambda v:float(np.sqrt(np.mean(v.dropna()**2))) if v.notna().any() else np.nan)
    ax.plot(errs.index,errs.values,marker='o',label=method)
ax.axvline(RELEASE_REFERENCE,color='black',ls='--',alpha=.5,label='visual release reference')
ax.set(title='Manual landmark reprojection comparison: detected frames only',
       xlabel='Original video frame',ylabel='2D point RMSE (pixels)')
ax.legend(fontsize=9);ax.grid(alpha=.2);fig.tight_layout()
fig.savefig(OUTPUT_DIR/'stage2j1_manual_landmark_rmse_by_frame.png',dpi=160)
plt.show()
assert hashlib.sha256(STAGE2H_CSV.read_bytes()).hexdigest()==source_hash,'Stage 2H source file changed!'
report={
 'stage':'2J-1','source_csv':str(STAGE2H_CSV),
 'source_sha256':source_hash,'source_unchanged':True,
 'video':str(VIDEO_PATH),'video_dimensions':[W,H],'fps':FPS,
 'frame_window':[FRAME_START,FRAME_END],'visual_release_reference':RELEASE_REFERENCE,
 'wilor_enabled':ENABLE_WILOR,
 'wilor_mano_license_acknowledged':I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS,
 'manual_roi_assisted_conditions':['mediapipe_manual_roi','wilor_manual_bbox'],
 'independent_conditions':['mediapipe_full','wilor_full_detector'],
 'assistance_warning':'Reference-derived ROI is not independent detection evaluation',
 '3d_warning':'Model 3D outputs use different coordinate frames, scale and origins; not ground truth.',
 'xy_warning':'Stage2H manual 2D points are not changed or overwritten.',
 'status_weights_note':'No fusion or point weighting performed in this comparison stage.'}
(OUTPUT_DIR/'stage2j1_run_report.json').write_text(
    json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('Output directory:',OUTPUT_DIR)
print('Source frozen SHA256:',source_hash)
print('Summary:',OUTPUT_DIR/'stage2j1_model_comparison_summary.csv')
print('Common table:',common_csv)
print('Stage 2H input unchanged.')

## Interpretation / next Stage 2J-2

- **Stage 2H is the reference 2D**, not a 3D ground truth; model predictions must not overwrite it.
- Independent full-frame vs assisted ROI conditions must be reported separately.
- WiLoR-mini depends on MANO model assets even though the package may download them automatically. **Check the license before enabling.**
- Compare frame 190's overlays and 2D error, detection rate and time-continuity. If the models diverge, inspect errors before 3D fusion.
- Next Stage 2J-2 will fit SOMA with 2D reprojection and temporal regularization, using only validated 3D initialization where compatible.

**Source**: [WiLoR-mini code](https://github.com/warmshao/WiLoR-mini) / [MediaPipe Tasks Hand Landmarker](https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker/python).